# Dumping code from `group-four-first-draft.ipynb` notebook that is no longer relevant.

In [ ]:
# Attempt to merge images_df with galaxyzoo based on dec and ra
zoo_df = pd.read_csv("data/zoo2MainSpecz.csv")

In [ ]:
zoo_df.head()

,specobjid,dr8objid,dr7objid,ra,dec,rastring,decstring,sample,gz2class,total_classifications,...,t11_arms_number_a36_more_than_4_fraction,t11_arms_number_a36_more_than_4_weighted_fraction,t11_arms_number_a36_more_than_4_debiased,t11_arms_number_a36_more_than_4_flag,t11_arms_number_a37_cant_tell_count,t11_arms_number_a37_cant_tell_weight,t11_arms_number_a37_cant_tell_fraction,t11_arms_number_a37_cant_tell_weighted_fraction,t11_arms_number_a37_cant_tell_debiased,t11_arms_number_a37_cant_tell_flag
0,1.802675e+18,NaN,588017703996096547,160.99040,11.703790,10:43:57.70,+11:42:13.6,original,SBb?t,44,...,0.225,0.225,0.225,0,10,10.0,0.250,0.250,0.250,0
1,1.992984e+18,NaN,587738569780428805,192.41083,15.164207,12:49:38.60,+15:09:51.1,original,Ser,45,...,0.000,0.000,0.000,0,0,0.0,0.000,0.000,0.000,0
2,1.489569e+18,NaN,587735695913320507,210.80220,54.348953,14:03:12.53,+54:20:56.2,original,Sc+t,46,...,0.651,0.651,0.651,0,3,3.0,0.070,0.070,0.070,0
3,2.924084e+18,1.237668e+18,587742775634624545,185.30342,18.382704,12:21:12.82,+18:22:57.7,original,SBc(r),45,...,0.071,0.071,0.071,0,6,6.0,0.429,0.429,0.429,0
4,1.387165e+18,1.237658e+18,587732769983889439,187.36679,8.749928,12:29:28.03,+08:44:59.7,extra,Ser,49,...,0.000,0.000,0.000,0,1,1.0,1.000,1.000,1.000,0


In [ ]:
zoo_df.columns.tolist()[:15]

['specobjid',
 'dr8objid',
 'dr7objid',
 'ra',
 'dec',
 'rastring',
 'decstring',
 'sample',
 'gz2class',
 'total_classifications',
 'total_votes',
 't01_smooth_or_features_a01_smooth_count',
 't01_smooth_or_features_a01_smooth_weight',
 't01_smooth_or_features_a01_smooth_fraction',
 't01_smooth_or_features_a01_smooth_weighted_fraction']

In [ ]:
zoo_features = [
    'dec',
    'ra',
    'rastring',
    'decstring',
    'gz2class'
]
 
zoo_df_small = zoo_df[zoo_features]
zoo_df_small.head()    

,dec,ra,rastring,decstring,gz2class
0,11.703790,160.99040,10:43:57.70,+11:42:13.6,SBb?t
1,15.164207,192.41083,12:49:38.60,+15:09:51.1,Ser
2,54.348953,210.80220,14:03:12.53,+54:20:56.2,Sc+t
3,18.382704,185.30342,12:21:12.82,+18:22:57.7,SBc(r)
4,8.749928,187.36679,12:29:28.03,+08:44:59.7,Ser


In [ ]:
zoo_dec = zoo_df_small['dec']
zoo_ra = zoo_df_small['ra']

In [ ]:
images_df_dec = images_df['dec']
images_df_ra = images_df['ra']

In [ ]:
# Use astropy to cross-match the two catalogs based on their celestial coordinates (RA and Dec) 
from astropy.coordinates import SkyCoord
from astropy import units as u

In [ ]:
sample_coords = SkyCoord(ra=images_df["ra"].values*u.degree, dec=images_df["dec"].values*u.degree)
zoo_coords = SkyCoord(ra=zoo_df["ra"].values*u.degree, dec=zoo_df["dec"].values*u.degree)

# Match galaxies to its nearest neighbor in the zoo catalog
idx, d2d, d3d = sample_coords.match_to_catalog_sky(zoo_coords)

# Filter out matches that are too far away
max_separation = 2.0 * u.arcsec  # Adjust this threshold based on the expected positional accuracy
matches = d2d < max_separation

# Create a DataFrame of matched galaxies
matched_df = pd.DataFrame({
    "image_idx": np.where(matches)[0],
    "zoo_idx": idx[matches],
    'image_ra': images_df['ra'].values[matches],
    'image_dec': images_df['dec'].values[matches],
    'zoo_ra': zoo_df['ra'].values[idx[matches]],
    'zoo_dec': zoo_df['dec'].values[idx[matches]],
    'separation': d2d[matches].arcsec
})

print(f"Number of matched galaxies: {len(matched_df)}")
print(matched_df.head())

Number of matched galaxies: 5538
   image_idx  zoo_idx    image_ra  image_dec     zoo_ra   zoo_dec  separation
0      29783   238605  173.293360  -1.488138  173.29337 -1.488162    0.096143
1      29788   109631  173.161750  -2.006866  173.16174 -2.006873    0.044243
2      29791   104581  173.144076  -1.910295  173.14406 -1.910292    0.059871
3      29799   239183  173.202903  -1.641754  173.20290 -1.641783    0.103360
4      29810   140939  174.724632  -2.089508  174.72462 -2.089508    0.044021


In [ ]:
# Pull only the zoo columns you want
zoo_labels = zoo_df[["gz2class"]].reset_index().rename(columns={"index": "zoo_idx"})

# Merge match info with zoo labels
matched_with_labels = matched_df.merge(zoo_labels, on="zoo_idx", how="left")

matched_with_labels.head()


,image_idx,zoo_idx,image_ra,image_dec,zoo_ra,zoo_dec,separation,gz2class
0,29783,238605,173.293360,-1.488138,173.29337,-1.488162,0.096143,Sc?t
1,29788,109631,173.161750,-2.006866,173.16174,-2.006873,0.044243,Sb2t
2,29791,104581,173.144076,-1.910295,173.14406,-1.910292,0.059871,Sc?m
3,29799,239183,173.202903,-1.641754,173.20290,-1.641783,0.103360,Sen
4,29810,140939,174.724632,-2.089508,174.72462,-2.089508,0.044021,Ser


In [ ]:
images_with_idx = images_df.reset_index().rename(columns={"index": "image_idx"})

images_labeled = images_with_idx.merge(
    matched_with_labels[["image_idx", "gz2class", "separation"]],
    on="image_idx",
    how="left",
)

images_labeled.head()


,image_idx,coord,dec,g_central_image_pop_10px_rad,g_central_image_pop_15px_rad,g_central_image_pop_5px_rad,g_cmodel_mag,g_cmodel_magsigma,g_ellipticity,g_half_light_radius,...,z_half_light_radius,z_isophotal_area,z_major_axis,z_minor_axis,z_peak_surface_brightness,z_petro_rad,z_pos_angle,z_sersic_index,gz2class,separation
0,0,"b'(179028.65625, 99644.3671875, -23767.86328125)'",-6.616883,1,1,1,20.162785,0.002381,0.026,2.085,...,2.186,127.0,2.112,1.951,-8.0189,4.62,63.61,1.146,NaN,NaN
1,1,"b'(178895.09375, 99811.6484375, -24069.6933593...",-6.701294,1,1,1,20.320715,0.005252,0.143,5.986,...,6.270,547.0,5.043,4.393,-7.6611,6.60,74.63,1.576,NaN,NaN
2,2,"b'(178918, 99992.578125, -23130.162109375)'",-6.438588,1,1,1,21.629736,0.013312,0.068,7.286,...,6.687,611.0,5.370,4.932,-7.2715,7.26,83.77,2.096,NaN,NaN
3,3,"b'(179111.03125, 99659.84375, -23072.220703125)'",-6.422391,1,1,1,21.448307,0.004646,0.012,2.108,...,2.966,121.0,2.323,2.269,-6.8408,6.60,29.82,1.943,NaN,NaN
4,4,"b'(178849.21875, 99990.828125, -23663.556640625)'",-6.587715,1,1,1,21.827169,0.010504,0.150,4.552,...,5.264,401.0,4.652,3.463,-7.2982,7.26,66.69,2.187,NaN,NaN


In [ ]:
images_labeled['gz2class'].value_counts()

gz2class
Er         870
Ei         817
Sb         435
Sc         427
Ser        333
          ... 
Sb3l         1
Sa(d)        1
Ec(o)        1
Sd?l(i)      1
Sb+m         1
Name: count, Length: 227, dtype: int64

In [ ]:
images_labeled_clean = images_labeled.dropna(subset=['gz2class'])

In [ ]:
images_labeled_clean.head()

,image_idx,coord,dec,g_central_image_pop_10px_rad,g_central_image_pop_15px_rad,g_central_image_pop_5px_rad,g_cmodel_mag,g_cmodel_magsigma,g_ellipticity,g_half_light_radius,...,z_half_light_radius,z_isophotal_area,z_major_axis,z_minor_axis,z_peak_surface_brightness,z_petro_rad,z_pos_angle,z_sersic_index,gz2class,separation
29783,29783,"b'(-204784.265625, 24080.68359375, -5356.69287...",-1.488138,1,1,1,17.735281,0.000706,0.472,14.519,...,13.928,2985.0,15.135,7.995,-8.2591,5.28,-41.58,1.007,Sc?t,0.096143
29788,29788,"b'(-204671.875, 24544.234375, -7223.24072265625)'",-2.006866,1,1,1,17.208687,0.000557,0.471,12.845,...,12.053,2145.0,12.136,6.760,-8.8767,5.94,-14.62,1.637,Sb2t,0.044243
29791,29791,"b'(-204676.09375, 24608.78515625, -6875.787109...",-1.910295,1,1,1,18.474453,0.001769,0.300,21.335,...,19.218,1949.0,12.582,9.017,-7.4077,8.58,46.19,2.157,Sc?m,0.059871
29799,29799,"b'(-204731, 24402.173828125, -5909.5068359375)'",-1.641754,1,1,1,17.853600,0.000837,0.669,18.452,...,17.046,2888.0,20.103,6.638,-7.4197,7.26,87.59,1.563,Sen,0.103360
29810,29810,"b'(-205254.5625, 18951.892578125, -7520.560546...",-2.089508,1,1,1,17.087647,0.000892,0.713,15.722,...,13.368,3561.0,19.923,6.080,-8.9362,5.28,38.69,1.563,Ser,0.044021


In [ ]:
images_labeled_clean.info()

<class 'pandas.DataFrame'>
Index: 5538 entries, 29783 to 204548
Data columns (total 87 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   image_idx                     5538 non-null   int64  
 1   coord                         5538 non-null   str    
 2   dec                           5538 non-null   float64
 3   g_central_image_pop_10px_rad  5538 non-null   int64  
 4   g_central_image_pop_15px_rad  5538 non-null   int64  
 5   g_central_image_pop_5px_rad   5538 non-null   int64  
 6   g_cmodel_mag                  5538 non-null   float64
 7   g_cmodel_magsigma             5538 non-null   float64
 8   g_ellipticity                 5538 non-null   float64
 9   g_half_light_radius           5538 non-null   float64
 10  g_isophotal_area              5538 non-null   float64
 11  g_major_axis                  5538 non-null   float64
 12  g_minor_axis                  5538 non-null   float64
 13  g_peak_surfac

In [ ]:
# Build a model-ready dataframe using only rows with matched Zoo labels
train_df = images_labeled_clean.copy()

# Turn text class labels into numeric labels for ML
train_df["gz2_label"], class_names = pd.factorize(train_df["gz2class"])

print("Training rows with labels:", len(train_df))
print("Unique classes:", len(class_names))
print("\nClass mapping:")
for i, c in enumerate(class_names):
    print(f"{i}: {c}")

# Example feature set
feature_cols = [
    "g_cmodel_mag",
    "r_cmodel_mag",
    "i_cmodel_mag",
    "z_cmodel_mag",
    "y_cmodel_mag",
    "g_sersic_index",
    "r_sersic_index",
    "i_sersic_index",
    "z_sersic_index",
    "y_sersic_index",
]

# Keep only features that actually exist in your dataframe
feature_cols = [c for c in feature_cols if c in train_df.columns]

X = train_df[feature_cols]
y = train_df["gz2_label"]

print("\nX shape:", X.shape)
print("y shape:", y.shape)


Training rows with labels: 5538
Unique classes: 227

Class mapping:
0: Sc?t
1: Sb2t
2: Sc?m
3: Sen
4: Ser
5: Sen(i)
6: Sc
7: Sc2t
8: Sc2m
9: SBd4l(i)
10: Sb(m)
11: Ei
12: SBb2m
13: Sb?t
14: Er
15: Sb
16: SBc2m
17: SBa2m
18: Sb(o)
19: Ei(o)
20: Sc(r)
21: SBb
22: SBd
23: Sc(i)
24: SBc2l
25: Sb?m
26: SBc
27: Er(o)
28: Sc3t
29: Sb1t
30: Sc1m
31: Sa?t
32: Sc4t
33: Sc(o)
34: Sd?m
35: Ec
36: SBb(i)
37: Sb(d)
38: Er(m)
39: Seb
40: Sc+t
41: SBc?m
42: Sb2m
43: Sc2l
44: Sc1t
45: Sc?t(i)
46: Sc+t(m)
47: Sc2m(o)
48: Sc1t(m)
49: Sc2l(m)
50: Sb1m
51: Ei(i)
52: Sc2m(d)
53: Sb3m(o)
54: SBb(r)
55: Sd(i)
56: Sb(r)
57: Sd
58: Sc2l(i)
59: SBc?t
60: Sb3m
61: Sc3m
62: Sb3t(r)
63: Sa(i)
64: Sb?t(m)
65: SBb2m(r)
66: Sc1m(m)
67: Sd2l
68: SBb2l
69: Sb2t(d)
70: Sc4m(d)
71: Sb3t(m)
72: SBb?t
73: Sb2m(i)
74: Sb+t
75: SBc3m
76: SBc3t
77: Sb?t(r)
78: Sc1l(i)
79: SBb2t
80: Sc3m(i)
81: Ei(m)
82: Ser(m)
83: SBc?l(i)
84: A
85: Sb?l
86: Sb2t(m)
87: SBc2t
88: Sc(d)
89: Sb1t(i)
90: Sd?t
91: SBb+t
92: Sc?l
93: Sb2t(r)
94: Sb